In [ ]:

# QRCL Memory Benchmark (Fixed Phase Logic + Noise Warnings Removed)

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
import numpy as np
import matplotlib.pyplot as plt

# Set up noise model once
def create_noise_model():
    noise_model = NoiseModel()
    depolar_error = depolarizing_error(0.02, 1)
    thermal_error = thermal_relaxation_error(70, 100, 1)
    noise_model.add_all_qubit_quantum_error(depolar_error, ['rx', 'rz'])
    noise_model.add_all_qubit_quantum_error(thermal_error, ['rx', 'rz'])
    return noise_model

# QRCL memory test function
def qrcl_memory_test(apply_phase_correction=False, apply_phase_drift=False, shots=1024, cycles=20):
    noise_model = create_noise_model()
    simulator = AerSimulator(noise_model=noise_model)
    survival_probs = []

    for _ in range(cycles):
        qc = QuantumCircuit(1, 1)
        qc.h(0)

        for _ in range(5):
            qc.rx(0.01, 0)  # induce noise with real gate

            if apply_phase_drift:
                qc.rz(0.015, 0)  # simulate ambient phase drift

            if apply_phase_correction:
                qc.rz(-0.015, 0)  # apply matching correction

        qc.h(0)
        qc.measure(0, 0)

        transpiled = transpile(qc, simulator)
        result = simulator.run(transpiled, shots=shots).result()
        counts = result.get_counts()
        prob_0 = counts.get('0', 0) / shots
        survival_probs.append(prob_0)

    return survival_probs

# Run benchmark cases
control = qrcl_memory_test(apply_phase_correction=False, apply_phase_drift=False)
drift_only = qrcl_memory_test(apply_phase_correction=False, apply_phase_drift=True)
corrected = qrcl_memory_test(apply_phase_correction=True, apply_phase_drift=True)

# Plot results
plt.figure(figsize=(10, 5))
plt.plot(control, label='No Drift / No Correction', marker='.')
plt.plot(drift_only, label='Drift Only', marker='x')
plt.plot(corrected, label='Drift + Phase Correction', marker='o')
plt.xlabel("QRCL Relay Cycle")
plt.ylabel("Probability of Measuring |0⟩")
plt.title("QRCL Memory Fidelity Over Time (Final Calibration)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
